In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
from flatten_json import flatten

In [3]:
# Automated date and time
start = dt.datetime(2019,4,18)
start = start.replace(hour=0, minute=0, second=0, microsecond=0)
#start = start - dt.timedelta(hours = 5, minutes = 30)
end = dt.datetime(2019,4,20)
end = end.replace(hour=0, minute=0, second=0, microsecond=0)
#end = end - dt.timedelta(hours = 5, minutes = 30)
print(start,end,end-start)

2019-04-18 00:00:00 2019-04-20 00:00:00 2 days, 0:00:00


In [4]:
base = '1970-01-01'
x = (start - pd.to_datetime(base)).total_seconds()
y = (end - pd.to_datetime(base)).total_seconds()
print(x)
print(y)

1555545600.0
1555718400.0


In [5]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query =( f"""SELECT
*
FROM
      `hitwicketsuperstars.analytics_190927423.new_user_reference`"""
       )
new_user_reference = client.query(query).to_dataframe()
new_user_reference.head()

,device_id,user_first_touch_timestamp,platform
0,998AE26B-478D-4DF7-A937-CEF65FDC572B,2019-05-29 09:12:35+00:00,IOS
1,9BF1D2A1-8942-45F4-B854-81AA27524D35,2019-05-29 17:25:09+00:00,IOS
2,B3CB8576-CFDB-40BA-915C-86ADC64FBF48,2019-05-29 17:07:31+00:00,IOS
3,2A0F1B8B-5079-43A6-A977-6C57F934BB52,2019-05-29 17:27:35+00:00,IOS
4,2AD82BEF-8880-4F64-8171-166F873684B0,2019-05-29 06:06:48+00:00,IOS


In [6]:
new_user_reference['user_first_touch_timestamp'] = new_user_reference['user_first_touch_timestamp'].astype('datetime64[s]')
android_new = new_user_reference[(new_user_reference['user_first_touch_timestamp'] >= start) & 
                                 (new_user_reference['user_first_touch_timestamp'] < end) & 
                                 (new_user_reference['platform'] == 'ANDROID')]

In [7]:
android_new.head()

,device_id,user_first_touch_timestamp,platform
1180,a93aac99096feb3184cc9db66bd2593b,2019-04-19 15:23:44,ANDROID
1664,51f11f73b77c260e3816c8d094166522,2019-04-19 09:46:12,ANDROID
1683,f38564585ac68c6fbf421487e5166c80,2019-04-19 18:01:08,ANDROID
2146,357e65ec5622e3d77140c8fba2a3fcbb,2019-04-19 11:49:28,ANDROID
2361,4a793ec8e89768dfeebdb46f90d60528,2019-04-19 14:40:08,ANDROID


In [8]:
android_new['user_first_touch_timestamp'] = ((android_new['user_first_touch_timestamp']-pd.to_datetime(base)).dt.total_seconds()).astype(int)

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


In [9]:
android_new.head()

,device_id,user_first_touch_timestamp,platform
1180,a93aac99096feb3184cc9db66bd2593b,1555687424,ANDROID
1664,51f11f73b77c260e3816c8d094166522,1555667172,ANDROID
1683,f38564585ac68c6fbf421487e5166c80,1555696868,ANDROID
2146,357e65ec5622e3d77140c8fba2a3fcbb,1555674568,ANDROID
2361,4a793ec8e89768dfeebdb46f90d60528,1555684808,ANDROID


In [10]:
len(android_new)

2998

In [11]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (f"""SELECT
            device_id,
            total_user_engagement,
            event_timestamp as session_timestamp
        FROM
            `hitwicketsuperstars.sessions.sessions_*` 
            WHERE (event_timestamp >= {x}*1000000 AND event_timestamp <= {y}*1000000)"""
        )
session = client.query(query).to_dataframe()
session.head()

,device_id,total_user_engagement,session_timestamp
0,6d152b53f446fddcadb09234eb709009,166.199,1555686573127000
1,67066f9ff8b3e4f5f24837f100104044,401.954,1555661534026000
2,d08226eced0dbe8191b7d431d3259882,14.929,1555684854952000
3,67066f9ff8b3e4f5f24837f100104044,213.239,1555639229466000
4,187fdab76a1172a3d413cbb823a66808,114.896,1555680527439000


In [12]:
len(session)

17314

In [13]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw_users = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}},{"sign_up_details":1, "created_at":1}):
    aw_users.append(documents)
dic_flattened = [flatten(d) for d in aw_users]
df_users = pd.DataFrame(dic_flattened)
users = df_users[["_id","created_at","sign_up_details_device_id"]]
users.columns = ["user_id","create_time","device_id"]

In [14]:
users = users.sort_values(['device_id','create_time'])
users = users.drop_duplicates('device_id')

In [15]:
users.head()

,user_id,create_time,device_id
24,5cb7e606733c6e6d0b7bdc32,2019-04-18 02:50:46.334,0003d212165a6fca57c4ceb569739e69
320,5cb8a7993aebfe688de01135,2019-04-18 16:36:41.178,000ac53b5ad9aeb9aaf0ddebdb011b30
846,5cb9ab453ed8807056a3a81b,2019-04-19 11:04:37.502,002a9f4473395bd52e653654dbf31bd3
683,5cb98e752bdfca07d8611ed9,2019-04-19 09:01:42.000,0064439812da373b34beb4f842e535a9
1219,5cba02de92f3047050b648f4,2019-04-19 17:18:22.277,00e82628815357fe4541b1de4a1a0bf7


In [16]:
len(users)

1302

In [17]:
user_session = pd.merge(session,users,on='device_id',how = 'inner')

In [18]:
user_session.head()

,device_id,total_user_engagement,session_timestamp,user_id,create_time
0,3c1889da344eb205097670a1584eb53f,51.425,1555655637637000,5cb94db39cde20080071ce64,2019-04-19 04:25:23.880
1,3c1889da344eb205097670a1584eb53f,786.472,1555660120598000,5cb94db39cde20080071ce64,2019-04-19 04:25:23.880
2,3c1889da344eb205097670a1584eb53f,791.969,1555640590400000,5cb94db39cde20080071ce64,2019-04-19 04:25:23.880
3,3c1889da344eb205097670a1584eb53f,2232.454,1555644841342000,5cb94db39cde20080071ce64,2019-04-19 04:25:23.880
4,1a216c4800869ddfe03dc73190d312c0,140.286,1555662630084000,5cb9dfd29cde200800749bb9,2019-04-19 14:48:50.893


In [19]:
len(user_session)

1917

In [20]:
user_session['create_time'] = ((user_session['create_time']-pd.to_datetime(base)).dt.total_seconds()).astype(int)*1000000

In [21]:
user_session.head()

,device_id,total_user_engagement,session_timestamp,user_id,create_time
0,3c1889da344eb205097670a1584eb53f,51.425,1555655637637000,5cb94db39cde20080071ce64,1555647923000000
1,3c1889da344eb205097670a1584eb53f,786.472,1555660120598000,5cb94db39cde20080071ce64,1555647923000000
2,3c1889da344eb205097670a1584eb53f,791.969,1555640590400000,5cb94db39cde20080071ce64,1555647923000000
3,3c1889da344eb205097670a1584eb53f,2232.454,1555644841342000,5cb94db39cde20080071ce64,1555647923000000
4,1a216c4800869ddfe03dc73190d312c0,140.286,1555662630084000,5cb9dfd29cde200800749bb9,1555685330000000


In [23]:
user_session = user_session[user_session['session_timestamp']-user_session['create_time']<(24*3600*1000000)]
user_session = user_session[user_session['session_timestamp'] > user_session['create_time']]

In [24]:
len(user_session)

443

In [25]:
user_session.head()

,device_id,total_user_engagement,session_timestamp,user_id,create_time
0,3c1889da344eb205097670a1584eb53f,51.425,1555655637637000,5cb94db39cde20080071ce64,1555647923000000
1,3c1889da344eb205097670a1584eb53f,786.472,1555660120598000,5cb94db39cde20080071ce64,1555647923000000
7,224792f1e02bd582a4abc482cbe5a93d,17.856,1555663935917000,5cb9665a9cde200800723592,1555654234000000
8,6ced2ff3e8763991ff0ab51184de31b0,1118.693,1555670356088000,5cb986839cde20080072fb04,1555662467000000
10,6ced2ff3e8763991ff0ab51184de31b0,2602.473,1555686649664000,5cb986839cde20080072fb04,1555662467000000


In [26]:
query = (
    f"""SELECT
  user_id,
  device.mobile_os_hardware_model as device
FROM `hitwicketsuperstars.analytics_190927423.events_2019*`
    WHERE _TABLE_SUFFIX BETWEEN '0417'
  AND '0420'
  GROUP BY user_id, device
      """
)
df_all = client.query(query).to_dataframe()

In [27]:
df_all.head(2)

,user_id,device
0,7e775b07cbd348b9a90d3b749e761b6d,vivo V3
1,32650426304eb118dcc84abec165426e,CPH1853


In [28]:
user_session_device = pd.merge(user_session[['device_id','total_user_engagement']],df_all,left_on='device_id',right_on='user_id',how='left')

In [29]:
len(user_session_device)

443

In [30]:
user_session_device.head()

,device_id,total_user_engagement,user_id,device
0,3c1889da344eb205097670a1584eb53f,51.425,3c1889da344eb205097670a1584eb53f,Redmi Note 4
1,3c1889da344eb205097670a1584eb53f,786.472,3c1889da344eb205097670a1584eb53f,Redmi Note 4
2,224792f1e02bd582a4abc482cbe5a93d,17.856,224792f1e02bd582a4abc482cbe5a93d,vivo 1603
3,6ced2ff3e8763991ff0ab51184de31b0,1118.693,6ced2ff3e8763991ff0ab51184de31b0,Micromax AQ5001
4,6ced2ff3e8763991ff0ab51184de31b0,2602.473,6ced2ff3e8763991ff0ab51184de31b0,Micromax AQ5001


In [31]:
len(user_session_device)

443

In [32]:
session_count = user_session_device.groupby('user_id')['total_user_engagement'].count().reset_index()
#g1 = df1.groupby( [ "Name", "City"] ).count().reset_index()

In [33]:
session_count.head()

,user_id,total_user_engagement
0,000ac53b5ad9aeb9aaf0ddebdb011b30,6
1,01acd72b9b33bbcd27688f5598dbc75e,1
2,0379eaae606d3b84d8e7140a8cefe001,1
3,043dd4b30246c9db3ac0a6dccfe75c55,1
4,0461c05e14f660b0fca8dc37c25fef9a,1


In [34]:
session_count['total_sessions'] = np.where(session_count['total_user_engagement']>=5,'5+',session_count['total_user_engagement'])
#B['temp'] = np.where(B['NOM']>=5, '5+', B['NOM'])

In [35]:
session_count.head()

,user_id,total_user_engagement,total_sessions
0,000ac53b5ad9aeb9aaf0ddebdb011b30,6,5+
1,01acd72b9b33bbcd27688f5598dbc75e,1,1
2,0379eaae606d3b84d8e7140a8cefe001,1,1
3,043dd4b30246c9db3ac0a6dccfe75c55,1,1
4,0461c05e14f660b0fca8dc37c25fef9a,1,1


In [44]:
len(session_count)

273

In [104]:
all_users = pd.merge(session_count,users[['device_id']],left_on='user_id',right_on='device_id',how='right')
all_users = all_users.fillna(0)
all_users.drop('user_id',inplace=True,axis=1)

In [129]:
final = all_users.groupby('total_sessions')['total_user_engagement'].count().reset_index()

In [130]:
final.set_index('total_sessions')

,total_user_engagement
total_sessions,
0,1029
1,181
2,54
3,19
4,11
5+,8


In [143]:
final['sum'] = [final.total_user_engagement[::-1][0]] + final.total_user_engagement[:0:-1].cumsum().values[::-1].tolist()

In [145]:
final.set_index('total_sessions',inplace=True)

In [146]:
final

,total_user_engagement,sum
total_sessions,,
0,1029,1029
1,181,273
2,54,92
3,19,38
4,11,19
5+,8,8
